# Tutorial: Finding, Accessing, and Comparing Data in DKRZ Waterpark

## Introduction

Waterpark is an interactive data exploration environment developed by DKRZ that simplifies access to climate and Earth system datasets. Rather than manually locating files on the HPC filesystem, users can browse datasets, inspect metadata, preview variables, and launch analyses.
A typical workflow consists of following steps:

- Find relevant datasets
- Access the data
- Process the data
- Compare datasets

The waterpark landing page allows you to explore datasets using metadata rather than filenames. If your datasets are published through a STAC (SpatioTemporal Asset Catalog) interface, the workflow becomes much more flexible because you discover datasets from metadata instead of browsing directories.


In [1]:
from pystac_client import Client
import xarray as xr
import fsspec
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

## Workflow

! Include Workflow Image and add step numbers

### Connect to the STAC Catalog

In [2]:
catalog = Client.open("https://freva.dkrz.de/api/freva-nextgen/stacapi/product/?visible_collections=cmip6%2Cdyamond%2Ceerie%2Cicdc%2Cicon-dream%2Cnextgems%2Cobs%2Cpalmod%2Creanalysis")
#catalog = Client.open("https://freva.dkrz.de/api/freva-nextgen/stacapi/product/?visible_collections=cmip6%2Cdyamond%2Ceerie%2Cicdc%2Cicon-dream%2Cnextgems%2Corchestra")

# What is the right API-Endpoint here? It keeps changing

### Find the Datasets

In [3]:
nextgems_search = catalog.search(
    collections=["nextgems"],
    datetime="2020-01-01/2020-12-31"
)

nextgems_item = next(nextgems_search.items())


In [4]:
nextgems_item

<Item id=1870876034525560832>

In [5]:
nextgems_item.assets["zarr-access"].href.split("=")[1]

'https://s3.waterpark.dkrz.de/nextgems/healpix/ngc4008/P1D/level_6.zarr'

**Link in STAC does not match what is needed for opening dataset!**

In [6]:
ds_nextgems = xr.open_zarr(fsspec.get_mapper("s3://nextgems/healpix/ngc4008/P1D/level_2.zarr", endpoint_url="https://s3.waterpark.dkrz.de", anon=True))

In [7]:
ds_era5 = xr.open_zarr(fsspec.get_mapper("s3://reanalysis/healpix/era5/P1D/level_2.zarr", endpoint_url="https://s3.waterpark.dkrz.de", anon=True))

### Select Variable

In [8]:
tas_nextgems = ds_nextgems["tas"]

In [9]:
tas_era5 = ds_era5["tas"]

### Match Time Period

In [10]:
tas_nextgems.time

<xarray.DataArray 'time' (time: 10958)> Size: 88kB
array(['2020-01-02T00:00:00.000000000', '2020-01-03T00:00:00.000000000',
       '2020-01-04T00:00:00.000000000', ..., '2049-12-30T00:00:00.000000000',
       '2049-12-31T00:00:00.000000000', '2050-01-01T00:00:00.000000000'],
      shape=(10958,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 88kB 2020-01-02 2020-01-03 ... 2050-01-01
Attributes:
    axis:     T

In [11]:
tas_era5.time

<xarray.DataArray 'time' (time: 31452)> Size: 252kB
array(['1940-01-01T12:00:00.000000000', '1940-01-02T12:00:00.000000000',
       '1940-01-03T12:00:00.000000000', ..., '2026-02-07T12:00:00.000000000',
       '2026-02-08T12:00:00.000000000', '2026-02-09T12:00:00.000000000'],
      shape=(31452,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 252kB 1940-01-01T12:00:00 ... 2026-02-09T1...
    crs      float64 8B ...
Attributes:
    standard_name:  time
    axis:           T

In [12]:
tas_nextgems_slice = tas_nextgems.sel(time=slice("2021-01-01", "2021-12-31"))

tas_era5_slice = tas_era5.sel(time=slice("2021-01-01", "2021-12-31"))

In [13]:
bias = tas_nextgems_slice - tas_era5_slice

In [14]:
bias

<xarray.DataArray 'tas' (time: 0, cell: 192)> Size: 0B
dask.array<sub, shape=(0, 192), dtype=float32, chunksize=(0, 192), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 0B 
  * cell       (cell) int64 2kB 0 1 2 3 4 5 6 7 ... 185 186 187 188 189 190 191
    latitude   (cell) float64 2kB dask.array<chunksize=(192,), meta=np.ndarray>
    longitude  (cell) float64 2kB dask.array<chunksize=(192,), meta=np.ndarray>
    crs        float64 8B ...
Attributes:
    cell_methods:   time: mean cell: mean
    component:      atmo
    grid_mapping:   crs
    standard_name:  air_temperature
    units:          K
    vgrid:          height_2m
    code:           167
    table:          128
    original_name:  var167